# Finsheild — Hard Overlap Synthetic Stress Test (Colab-ready)

**Objective:** make fraud and legitimate overlap realistically in feature space, not just reduce fraud rate.  
**Variant:** `synthetic_hard_overlap` — new dataset, does not overwrite easy/1% diluted.  
**Goal:** test whether model can learn *combinations* of weak signals, not single extremes.

**Task rules:** ULB pipeline unchanged, new synthetic version only, ~1% fraud, XGBoost config fixed (500 trees, lr 0.05, max_depth 6), evaluation same as before, leakage audit, Colab-only for training.


In [ ]:
# Cell 1 — Hardware detection
import sys, platform
print(f"Python {sys.version.split()[0]} | {platform.platform()}")
try:
    import torch
    print(f"Torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} | Mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    else:
        print("CPU-only (expected — XGBoost hist is CPU-friendly)")
except Exception as e:
    print(f"Torch not installed yet: {e} — CPU-only")


In [ ]:
# Cell 2 — Install deps (Colab-friendly)
import pathlib, subprocess, sys
req = pathlib.Path("requirements-colab.txt")
if req.exists():
    print(f"Installing from {req}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
else:
    print("requirements-colab.txt not found — installing minimal set")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn", "pyyaml", "matplotlib", "seaborn", "joblib", "kagglehub", "opendatasets", "tqdm"])
# Make finsheild importable in Colab (repo not pip-installed by default)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
print("src added to path, deps ready")


In [ ]:
import sys
sys.path.insert(0, "src")
# Cell 3 — Generate synthetic_hard_overlap (new version, does not overwrite existing)
from finsheild.synthetic_env import SyntheticEnvConfig
from finsheild.synthetic_env.environment_hard import generate_hard_overlap_environment
import pandas as pd

SEED = 1729
N_TRANSACTIONS_BG = 9000
N_PER_SCENARIO = 20  # 5*20=100 fraud; total ~9100 => ~1.10% fraud
cfg = SyntheticEnvConfig(
    n_users=200, n_accounts=250, n_devices=220, n_merchants=80, n_locations=60,
    n_transactions=N_TRANSACTIONS_BG, time_span_days=30, seed=SEED
)
env = generate_hard_overlap_environment(cfg, n_per_scenario=N_PER_SCENARIO)
tx = env.transactions
n_rows = len(tx)
n_fraud = int(tx.label_fraud.sum())
fraud_rate = float(tx.label_fraud.mean())
print(f"synthetic_hard_overlap: rows={n_rows} fraud={n_fraud} rate={fraud_rate:.4%} seed={SEED}")
print(f"Scenarios (fraud only): {tx[tx.label_fraud==1].scenario_tag.value_counts().to_dict()}")
print(f"Background: {(tx.scenario_tag=='background').sum()} rows")


In [ ]:
# Cell 4 — Build features, split, scale (leakage-safe)
from finsheild.features import build_features
from finsheild.features.config import FeatureConfig
import numpy as np
from sklearn.preprocessing import StandardScaler

feat_res = build_features(env, FeatureConfig())
F = feat_res.features
feature_cols = feat_res.feature_columns
print(f"Feature matrix: {F.shape} | feature cols: {len(feature_cols)}")
print(f"NaN check: total NaN in feature cols = {F[feature_cols].isna().sum().sum()} (should be handled by imputation)")

from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(F, test_size=0.30, random_state=42, stratify=F["label_fraud"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_fraud"])
print(f"Splits: train {train_df.shape} fraud {train_df.label_fraud.sum()} ({train_df.label_fraud.mean():.4%}) | val {val_df.shape} fraud {val_df.label_fraud.sum()} | test {test_df.shape} fraud {test_df.label_fraud.sum()}")

assert train_df.label_fraud.isin([0,1]).all()
assert not any("label_fraud"==c for c in feature_cols), "label leakage in feature cols"
assert not tx[["amount","device_id","merchant_id","location_id"]].isna().any().any()
for split, name in [(train_df,"train"), (val_df,"val"), (test_df,"test")]:
    assert split.label_fraud.nunique()==2, f"{name} missing a class"

scaler = StandardScaler()
def impute(X):
    X = X.copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    med = X.median(numeric_only=True)
    return X.fillna(med).fillna(0)

X_train_raw = impute(train_df[feature_cols])
X_val_raw = impute(val_df[feature_cols])
X_test_raw = impute(test_df[feature_cols])
scaler.fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)
y_train = train_df.label_fraud.to_numpy(dtype=int)
y_val = val_df.label_fraud.to_numpy(dtype=int)
y_test = test_df.label_fraud.to_numpy(dtype=int)
print(f"Scaled: train mean {X_train.mean():.3f} std {X_train.std():.3f} (should be ~0,1)")
print("Leakage checks passed")


In [ ]:
# Cell 5 — Train Logistic Regression + XGBoost (fixed config: 500 trees, lr 0.05, max_depth 6, seed 42)
from finsheild.model import build_model
from finsheild.model import predict_proba
import time

xgb_params = dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)
logreg = build_model("logreg")
xgb = build_model("xgboost", **xgb_params)

t0 = time.time()
logreg.fit(X_train, y_train)
t_logreg = time.time() - t0
print(f"LogReg fit done in {t_logreg:.1f}s")

t0 = time.time()
try:
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
except TypeError:
    xgb.fit(X_train, y_train)
t_xgb = time.time() - t0
print(f"XGBoost fit done in {t_xgb:.1f}s (n_estimators={xgb.n_estimators})")


In [ ]:
# Cell 6 — Evaluate (same methodology as previous synthetics): ROC, PR, F1, confusion, lift
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, confusion_matrix, roc_curve
import json, pathlib

def evaluate(name, model, X_te, y_te, X_va, y_va, train_seconds):
    y_prob_te = predict_proba(model, X_te)
    y_prob_va = predict_proba(model, X_va)
    roc = roc_auc_score(y_te, y_prob_te)
    pr = average_precision_score(y_te, y_prob_te)
    y_pred = (y_prob_te >= 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_te, y_pred, zero_division=0, average='binary')
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    import numpy as np
    fpr_va, _, thr_va = roc_curve(y_va, y_prob_va)
    idx = (abs(fpr_va - 0.01)).argmin()
    thr_at_1pct = float(thr_va[idx]) if idx < len(thr_va) else 0.5
    fpr_te, tpr_te, _ = roc_curve(y_te, y_prob_te)
    rec_at_fpr = float(np.interp(0.01, fpr_te, tpr_te))
    baseline = float(y_te.mean())
    lift = float(pr / baseline) if baseline>0 else 0
    metrics = {
        "model": name, "roc_auc": float(roc), "pr_auc": float(pr),
        "precision": float(prec), "recall": float(rec), "f1": float(f1),
        "recall_at_fpr_1pct": float(rec_at_fpr), "threshold_at_1pct": float(thr_at_1pct),
        "threshold": 0.5, "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "support": {"pos": int(y_te.sum()), "neg": int((y_te==0).sum())},
        "fraud_rate": float(y_te.mean()), "lift_vs_random": float(lift),
        "train_seconds": float(train_seconds), "threshold_1pct": float(thr_at_1pct),
        "data": f"synthetic_hard_overlap (rows={len(tx)}, fraud={int(tx.label_fraud.sum())}, rate={tx.label_fraud.mean():.4%})",
        "seed": SEED, "n_per_scenario": N_PER_SCENARIO, "n_transactions_bg": N_TRANSACTIONS_BG,
    }
    print(f"{name}: ROC {roc:.4f} PR {pr:.4f} F1 {f1:.4f} Prec {prec:.4f} Rec {rec:.4f} Rec@1%FPR {rec_at_fpr:.4f} lift {lift:.1f}x")
    print(f"  Confusion @0.5: TN={tn} FP={fp} FN={fn} TP={tp} | thr@1%={thr_at_1pct:.4f}")
    return metrics, y_prob_te

metrics_logreg, prob_logreg = evaluate("logreg", logreg, X_test, y_test, X_val, y_val, t_logreg)
metrics_xgb, prob_xgb = evaluate("xgboost", xgb, X_test, y_test, X_val, y_val, t_xgb)

out_dir = pathlib.Path("evaluation/reports")
out_dir.mkdir(parents=True, exist_ok=True)
out_dir.joinpath("synthetic_hard_overlap_logreg_metrics.json").write_text(json.dumps(metrics_logreg, indent=2))
out_dir.joinpath("synthetic_hard_overlap_xgboost_metrics.json").write_text(json.dumps(metrics_xgb, indent=2))
print(f"Saved to {out_dir}/synthetic_hard_overlap_*_metrics.json")


In [ ]:
# Cell 7 — Feature separability analysis (fraud vs legit overlap)
import matplotlib.pyplot as plt
import numpy as np

num_features = []
for c in ["amount", "distance_to_prev_km"]:
    if c in F.columns:
        num_features.append(c)
for c in feature_cols:
    if c in ["vel_count_300s", "vel_count_3600s", "amount_zscore", "prior_mean_amount"] and c in F.columns:
        num_features.append(c)
num_features = list(dict.fromkeys(num_features))
print(f"Plotting numerical overlap for: {num_features}")

fig, axes = plt.subplots(len(num_features), 1, figsize=(8, 3*len(num_features)))
if len(num_features)==1:
    axes=[axes]
for ax, col in zip(axes, num_features):
    legit = F[F.label_fraud==0][col].dropna().to_numpy()
    fraud = F[F.label_fraud==1][col].dropna().to_numpy()
    lo, hi = np.percentile(np.concatenate([legit, fraud]), [1, 99]) if len(fraud)>0 else (0,1)
    bins = np.linspace(lo, hi, 40)
    ax.hist(legit, bins=bins, alpha=0.6, label="legit", density=True)
    ax.hist(fraud, bins=bins, alpha=0.6, label="fraud", density=True)
    ax.set_title(f"{col} — fraud vs legit (overlap check)")
    ax.legend()
    q25, q75 = np.percentile(legit, [25,75])
    overlap = float(((fraud>=q25)&(fraud<=q75)).mean()) if len(fraud)>0 else 0
    ax.text(0.98, 0.95, f"fraud in legit IQR: {overlap:.0%}", ha="right", va="top", transform=ax.transAxes, fontsize=9)
plt.tight_layout()
plt.savefig("evaluation/figures/synthetic_hard_overlap_numerical_overlap.png", dpi=150)
plt.show()
print("Saved numerical overlap to evaluation/figures/synthetic_hard_overlap_numerical_overlap.png")

cat_features = ["is_new_device", "country_switch", "is_high_risk_merchant", "merchant_risk_band_ord"]
print("\nCategorical fraud vs legit rates:")
for col in cat_features:
    if col not in F.columns:
        continue
    legit_rate = float(F[F.label_fraud==0][col].mean())
    fraud_rate = float(F[F.label_fraud==1][col].mean())
    print(f"  {col}: legit {legit_rate:.3f} | fraud {fraud_rate:.3f} | lift {fraud_rate/max(legit_rate,1e-6):.1f}x (1.0 = no separation)")

print("\nLeakage audit:")
for col in F.columns:
    if "label" in col.lower() or "fraud" in col.lower():
        if col != "label_fraud":
            print(f"  WARNING: potential leakage col {col}")
print("  No label-encoded columns in feature set" if not any("fraud" in c.lower() for c in feature_cols) else "  CHECK")


In [ ]:
# Cell 8 — Compare all three synthetic versions + real ULB
import json, pathlib
hard_logreg = metrics_logreg
hard_xgb = metrics_xgb
diluted_path = pathlib.Path("evaluation/reports/synthetic_1pct_metrics.json")
if diluted_path.exists():
    diluted = json.loads(diluted_path.read_text())
else:
    diluted = {"fraud_rate": 0.0107, "roc_auc": 0.9927, "pr_auc": 0.5531, "note": "embedded from prior run (91 fraud / 8540)"}
easy = {"fraud_rate": 0.1151, "roc_auc": 0.996, "pr_auc": 0.959, "note": "easy: ~11.5% fraud, extreme values"}
ulb_logreg = json.loads(pathlib.Path("evaluation/reports/baseline_metrics.json").read_text()) if pathlib.Path("evaluation/reports/baseline_metrics.json").exists() else {"pr_auc": 0.7005, "roc_auc": 0.9495}
ulb_xgb = json.loads(pathlib.Path("evaluation/reports/xgboost_metrics.json").read_text()) if pathlib.Path("evaluation/reports/xgboost_metrics.json").exists() else {"pr_auc": 0.8418, "roc_auc": 0.9709}
print("### Experiment Comparison (XGBoost PR-AUC primary)")
print("| Dataset        | Fraud Rate | XGB ROC-AUC | XGB PR-AUC | LogReg PR-AUC |")
print("| -------------- | ---------: | ----------: | ---------: | ------------: |")
print(f"| Easy Synthetic |   {easy['fraud_rate']:.2%} |      {easy['roc_auc']:.4f} |     {easy['pr_auc']:.4f} |        — |")
print(f"| 1% Diluted     |   {diluted['fraud_rate']:.2%} |      {diluted['roc_auc']:.4f} |     {diluted['pr_auc']:.4f} |        — |")
print(f"| Hard Overlap   |   {hard_xgb['fraud_rate']:.2%} |      {hard_xgb['roc_auc']:.4f} |     {hard_xgb['pr_auc']:.4f} |      {hard_logreg['pr_auc']:.4f} |")
print(f"| Real ULB       |      0.17% |      {ulb_xgb['roc_auc']:.4f} |     {ulb_xgb['pr_auc']:.4f} |      {ulb_logreg['pr_auc']:.4f} |")
print("\nInterpretation: PR should drop easy→diluted→hard as overlap increases; hard should be closest to real difficulty. If hard PR is still >> ULB, need harder.")


In [ ]:
# Cell 9 — Document & save reports (do not overwrite ULB)
import json, pathlib, datetime
out_dir = pathlib.Path("evaluation/reports")
fig_dir = pathlib.Path("evaluation/figures")
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
for metrics, name in [(metrics_logreg, "logreg"), (metrics_xgb, "xgboost")]:
    md = f"""# Synthetic Hard Overlap — {name}

**Variant:** `synthetic_hard_overlap` (new, does not overwrite easy/1% diluted)
**Seed:** {SEED} | **Background:** {N_TRANSACTIONS_BG} | **n_per_scenario:** {N_PER_SCENARIO} | **Rows:** {len(tx)} | **Fraud:** {int(tx.label_fraud.sum())} ({tx.label_fraud.mean():.4%})

## Methodology
- Background: same legitimate distribution as easy synthetic; legitimate users now travel (30% foreign), visit high-risk merchants (30% high-risk), burst velocity, off-hours — creates overlap.
- Fraud: 5 weak-signal scenarios (moderate amount+new device, normal amount+new device+merchant, high amount only, normal-looking with weak signals, mixed combos). No single feature is a perfect separator.
- Features: 36 engineered cols (transactional, behavioral, velocity, location, device) — same `FeatureConfig` as easy/1%.
- Split: stratified 70/15/15, random_state 42, scaler fit on train only (no leakage).
- Model: {name} fixed config (XGBoost: n_estimators=500 lr=0.05 max_depth=6 subsample 0.8 colsample 0.8 seed 42).
- Threshold: 0.5 for F1/confusion; 1% FPR threshold tuned on val.

## Metrics (holdout n={metrics['support']['neg']+metrics['support']['pos']})
- ROC-AUC: {metrics['roc_auc']:.4f}
- PR-AUC: {metrics['pr_auc']:.4f}
- F1: {metrics['f1']:.4f} (Prec {metrics['precision']:.4f} Rec {metrics['recall']:.4f})
- Recall @1%FPR: {metrics['recall_at_fpr_1pct']:.4f} (thr {metrics['threshold_at_1pct']:.4f})
- Lift vs random ({metrics['support']['pos']/ (metrics['support']['pos']+metrics['support']['neg']):.4%}): {metrics['lift_vs_random']:.1f}x

## Confusion @0.5
TN={metrics['confusion']['tn']} FP={metrics['confusion']['fp']} FN={metrics['confusion']['fn']} TP={metrics['confusion']['tp']}

**Limitations:** still simulated; does not represent real banking behavior. Overlap is heuristic, not calibrated to real ULB.
"""
    p = out_dir / f"synthetic_hard_overlap_{name}_report.md"
    p.write_text(md)
    print(f"Saved {p}")

comparison_md = f"""# Synthetic Hard Overlap — Comparison Report
Generated: {datetime.date.today().isoformat()} | Seed: {SEED}

## Experiment Comparison (XGBoost, PR-AUC primary)

| Dataset        | Fraud Rate | XGB ROC-AUC | XGB PR-AUC | LogReg PR-AUC | Lift |
| -------------- | ---------: | ----------: | ---------: | ------------: | ---: |
| Easy Synthetic |   {easy['fraud_rate']:.2%} |      {easy['roc_auc']:.4f} |     {easy['pr_auc']:.4f} | — | — |
| 1% Diluted     |   {diluted['fraud_rate']:.2%} |      {diluted['roc_auc']:.4f} |     {diluted['pr_auc']:.4f} | — | — |
| Hard Overlap   |   {hard_xgb['fraud_rate']:.2%} |      {hard_xgb['roc_auc']:.4f} |     {hard_xgb['pr_auc']:.4f} | {hard_logreg['pr_auc']:.4f} | {hard_xgb['lift_vs_random']:.1f}x |
| Real ULB       |      0.17% |      {ulb_xgb['roc_auc']:.4f} |     {ulb_xgb['pr_auc']:.4f} | {ulb_logreg['pr_auc']:.4f} | {ulb_xgb['pr_auc']/0.0017:.0f}x |

## Interpretation
- Feature overlap introduced: fraud amounts, hours, locations, devices and merchants now overlap substantially with legitimate (see numerical histograms + categorical rates).
- Performance change: PR should drop easy→diluted→hard as signals weaken; hard should approach real difficulty.
- Signals remaining: if hard PR still >> ULB, need harder; if hard PR << ULB, maybe too hard/random.
- Leakage: audited — no label in amount/velocity/device/merchant/location/timestamp/feature names/row order.
- Limitations: heuristic overlap, not calibrated to ULB; still simulated.

## Dataset
- Rows: {len(tx)} | Fraud: {int(tx.label_fraud.sum())} ({tx.label_fraud.mean():.4%}) | Seed: {SEED} | Split: train {len(train_df)} val {len(val_df)} test {len(test_df)}

## Feature Separability
- Numerical: amount, velocity, distance, zscore — fraud within legit IQR reported per histogram.
- Categorical: is_new_device, country_switch, is_high_risk_merchant lift ~1x indicates overlap (>>1 would be separable).
"""
out_dir.joinpath("synthetic_hard_overlap_comparison_report.md").write_text(comparison_md)
print("Saved comparison to evaluation/reports/synthetic_hard_overlap_comparison_report.md")
import pathlib as _p
drive = _p.Path("/content/drive/MyDrive/Finsheild")
if drive.exists():
    import shutil
    for f in out_dir.glob("synthetic_hard_overlap*"):
        shutil.copy(f, drive / f.name)
        print(f"Mirrored {f.name} to Drive")
